# 🧬 MTP の要求

draft model なしで mtp が成立する条件とファイル探索。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

import json

ROOT = find_root()

t = (root / "config/speculative.py").read_text(errors="replace")
lines = t.splitlines()

print("=== __post_init__ の mtp 分岐（前後）===")
for i, l in enumerate(lines, 1):
    if '"mtp"' in l and ("method" in l or "==" in l):
        a, b = max(0, i - 20), min(len(lines), i + 90)
        for j in range(a, b):
            print(f"  {j + 1}: {lines[j][:170]}")
        break

print("\n=== qwen3_5 固有の分岐 ===")
for i, l in enumerate(lines, 1):
    if "qwen3_5" in l:
        a, b = max(0, i - 8), min(len(lines), i + 30)
        print(f"  --- line {i} ---")
        for j in range(a, b):
            print(f"  {j + 1}: {lines[j][:170]}")
        break

print("\n=== 必須項目の検証 ===")
for i, l in enumerate(lines, 1):
    s = l.strip()
    if ("raise" in s or "assert" in s) and ("spec" in s.lower() or "draft" in s.lower()
                                            or "mtp" in s.lower() or "model" in s.lower()):
        print(f"  {i}: {s[:170]}")

print("\n=== safetensors の探索（loader）===")
for f in ("model_executor/model_loader/weight_utils.py",
          "model_executor/model_loader/default_loader.py",
          "model_executor/model_loader/loader.py"):
    p = root / f
    if not p.exists():
        continue
    tt = p.read_text(errors="replace")
    for i, l in enumerate(tt.splitlines(), 1):
        s = l.strip()
        if "safetensors" in s and ("glob" in s or "endswith" in s or "iter" in s):
            print(f"  {f}:{i}: {s[:160]}")

print("\n=== config.json の mtp 関連 ===")
cfg = json.load(open(ROOT + "/config.json"))
for k in sorted(cfg):
    if "mtp" in k.lower() or "layer" in k.lower() or "hidden" in k.lower():
        v = cfg[k]
        print(f"  {k}: {v if not isinstance(v, (dict, list)) else type(v).__name__}")
tc = cfg.get("text_config") or {}
for k in sorted(tc):
    if "mtp" in k.lower() or "layer" in k.lower():
        print(f"  text_config.{k}: {tc[k]}")

print("\n=== draft クラス（Qwen3_5MTP）の前提 ===")
m = (root / "model_executor/models/qwen3_5_mtp.py").read_text(errors="replace")
for i, l in enumerate(m.splitlines(), 1):
    s = l.strip()
    if s.startswith("class ") or "num_mtp_layers" in s or "mtp_start_layer_idx" in s:
        print(f"  {i}: {s[:160]}")
